# Kafka & Streaming 面试知识点整理


## 1. Kafka Architecture / Kafka 架构

### 🎤 English Answer (30-40s)

Kafka is a distributed event streaming platform built around a few core components. **Producers** publish messages to **topics**, which are split into **partitions** for parallelism and scalability. Each partition is an ordered, immutable append-only log. Partitions are replicated across multiple **brokers** for fault tolerance — one replica is the **leader** handling all reads and writes, and the others are **followers** that replicate data. **Consumers** read from partitions, and they're organized into **consumer groups** where each partition is assigned to exactly one consumer in the group, enabling parallel consumption. Metadata and coordination used to be managed by **ZooKeeper**, but Kafka is now migrating to **KRaft** mode for self-managed metadata.

### 📖 中文详解

Kafka 的核心架构可以从五个层面理解：

**Broker（代理/节点）**：Kafka cluster 由多个 broker 组成，每个 broker 是一个独立的服务器进程，负责存储数据和处理请求。Broker 之间没有 master-slave 的固定关系（KRaft 模式下有 controller），它们是对等的。

**Topic（主题） & Partition（分区）**：Topic 是逻辑上的消息分类（比如 "user-clicks"），但实际数据存储在 partition 里。一个 topic 可以有多个 partition，每个 partition 是一个有序的、不可变的、只能追加写入（append-only）的 log 文件。Partition 是 Kafka 并行处理的最小单位——partition 数量决定了消费端最大并行度。

**Replication（副本机制）**：每个 partition 有一个 leader replica 和若干个 follower replica，分布在不同 broker 上。所有读写请求都由 leader 处理，follower 只负责从 leader 拉取数据保持同步。这个设计保证了即使某个 broker 宕机，数据也不会丢失。

**Producer & Consumer**：Producer 决定把消息发到哪个 partition（通过 key hash 或 round-robin），Consumer 通过 consumer group 实现并行消费。同一个 group 内，每个 partition 只会被一个 consumer 消费（保证顺序），不同 group 之间互不影响（实现 pub/sub 模型）。

**ZooKeeper / KRaft**：传统架构依赖 ZooKeeper 来管理 broker 注册、leader 选举、topic 配置等元数据。KRaft 模式把这些功能内化到 Kafka 自身。

---



## 2. ISR (In-Sync Replicas)

### 🎤 English Answer (30-40s)

ISR stands for In-Sync Replicas. It's the set of replicas — including the leader — that are fully caught up with the leader's log. A follower stays in the ISR as long as it fetches data from the leader within a configurable time window, controlled by `replica.lag.time.max.ms`. If a follower falls behind, it gets removed from the ISR. The ISR is critical for durability: when a producer sets `acks=all`, the leader only acknowledges a write after all replicas in the ISR have received it. If the leader fails, only replicas in the ISR are eligible to become the new leader, which prevents data loss.

### 📖 中文详解

**ISR 的定义**：ISR 是当前与 leader 保持同步的 replica 集合。注意 leader 自己也在 ISR 里。

**如何判断"同步"**：Kafka 用 `replica.lag.time.max.ms`（默认 30 秒）来判断。只要 follower 在这个时间窗口内成功从 leader fetch 过数据，就算"in-sync"。如果超过这个时间没有 fetch（可能是网络抖动、broker 过载、GC 暂停等原因），该 follower 就会被踢出 ISR。

**ISR 的意义**：
- **写入确认**：当 producer 配置 `acks=all` 时，leader 要等到 ISR 中所有 replica 都写入成功才返回 ack。ISR 越大，数据越安全，但延迟也越高。
- **Leader 选举**：只有 ISR 中的 follower 才有资格被选为新 leader（除非开启 `unclean.leader.election.enable`，但这可能丢数据）。
- **动态变化**：ISR 不是固定的，会随着 follower 的同步状态动态调整。这个信息存储在 ZooKeeper（或 KRaft 的元数据 log）中。

**面试延伸**：如果 ISR 缩小到只剩 leader，`acks=all` 等价于 `acks=1`，此时数据安全性下降。可以通过 `min.insync.replicas` 来设置最低 ISR 数量，低于这个值时 producer 写入会失败，从而保护数据不被写入不安全的状态。

---



## 3. Leader Election / Leader 选举

### 🎤 English Answer (30-40s)

When a partition leader fails, Kafka's controller is responsible for electing a new leader. The controller picks a new leader from the ISR — the set of replicas that are fully caught up. Typically, the first replica in the ISR list is chosen. If the ISR is empty and `unclean.leader.election.enable` is set to true, Kafka can elect an out-of-sync replica, but this risks data loss. In ZooKeeper mode, one broker acts as the controller by acquiring a ZooKeeper lock. In KRaft mode, the controller is elected via Raft consensus among a set of controller nodes, which eliminates the ZooKeeper dependency.

### 📖 中文详解

**谁来执行 leader 选举？** Controller。在整个 Kafka cluster 中，有一个 broker 被选为 controller，负责管理所有 partition 的 leader 选举。

**选举触发条件**：
- Leader broker 宕机或失联
- Broker 正常关闭（graceful shutdown，会主动把 leadership 移交）
- 手动触发 partition reassignment

**选举过程**：
1. Controller 检测到某个 partition 的 leader 不可用
2. 从该 partition 的 ISR 列表中选择第一个可用的 replica 作为新 leader
3. 更新元数据并通知所有相关 broker

**Clean vs Unclean Leader Election**：
- **Clean**（默认）：只从 ISR 中选 leader，保证数据一致性，但如果 ISR 为空，该 partition 暂时不可用
- **Unclean**（`unclean.leader.election.enable=true`）：允许从非 ISR 的 follower 中选 leader，保证可用性但可能丢数据（因为该 follower 没有完全同步）

**Controller 本身的选举**：
- ZooKeeper 模式：多个 broker 竞争创建 ZooKeeper 的临时节点 `/controller`，先创建成功的成为 controller
- KRaft 模式：通过 Raft 协议在 controller quorum 节点中选举 leader，不再依赖外部系统

---



## 4. At-Least-Once vs Exactly-Once Semantics

### 🎤 English Answer (30-40s)

**At-least-once** means every message is guaranteed to be delivered, but duplicates are possible. This happens when a producer sends a message, the broker persists it, but the ack is lost — the producer retries and the message is written twice. To achieve **exactly-once**, Kafka uses two mechanisms: **idempotent producers** and **transactions**. Idempotent producers assign a sequence number to each message so the broker can deduplicate retries. Transactions allow atomic writes across multiple partitions — either all messages in a transaction are committed or none are. On the consumer side, exactly-once requires using the transactional API with `read_committed` isolation level, so consumers only see committed messages.

### 📖 中文详解

**三种语义**：
- **At-most-once**：消息可能丢失，不会重复。Producer 发完就不管了（`acks=0`），或者 consumer 先 commit offset 再处理消息。
- **At-least-once**：消息不会丢失，但可能重复。Producer 用 `acks=all` + 重试；consumer 先处理消息再 commit offset。
- **Exactly-once**：消息既不丢失也不重复。

**Exactly-once 在 Producer 端的实现**：

Idempotent Producer（幂等生产者）：
- 开启 `enable.idempotence=true`
- 每个 producer 获得一个唯一的 Producer ID (PID)
- 每条消息携带一个 sequence number
- Broker 端会记录每个 `<PID, Partition>` 对应的最大 sequence number
- 如果收到重复的 sequence number，broker 直接丢弃，不会重复写入
- 注意：幂等性只保证单个 partition 内的 exactly-once，跨 partition 需要 transaction

Transactional Producer（事务生产者）：
- 通过 `transactional.id` 配置唯一标识
- 支持原子性地写入多个 partition
- 流程：`beginTransaction()` → `send()` → `commitTransaction()` 或 `abortTransaction()`
- Broker 端有一个 Transaction Coordinator 来管理事务状态

**Exactly-once 在 Consumer 端的实现**：
- 设置 `isolation.level=read_committed`，consumer 只读取已提交的事务消息
- 配合 "consume-transform-produce" 模式，把 offset commit 和下游写入放在同一个事务里

**面试关键点**：在大多数实际场景中，at-least-once + 下游幂等处理（如 upsert、deduplication）是最常见的做法，因为 exactly-once 有性能开销。

---



## 5. Offset Commit / Offset 提交机制

### 🎤 English Answer (30-40s)

Offsets track the position of a consumer in a partition. Kafka stores committed offsets in an internal topic called `__consumer_offsets`. There are two commit modes: **auto-commit** and **manual commit**. With auto-commit enabled, the consumer periodically commits the latest offset in the background, controlled by `auto.commit.interval.ms`. But if the consumer crashes between commits, messages may be reprocessed. Manual commit gives more control — you can call `commitSync()` for synchronous, blocking commits, or `commitAsync()` for non-blocking ones. A common pattern is to commit after successfully processing a batch of messages to ensure at-least-once semantics.

### 📖 中文详解

**Offset 是什么**：Offset 是每条消息在 partition 中的唯一递增序号（从 0 开始）。Consumer 通过 offset 来标记"我读到哪了"。

**存储位置**：Committed offset 存储在 Kafka 内部 topic `__consumer_offsets` 中（默认 50 个 partition）。以前存在 ZooKeeper 里，现在已经迁移到 Kafka 内部。

**两种提交模式**：

| 模式 | 配置 | 特点 |
|------|------|------|
| Auto Commit | `enable.auto.commit=true` | 每隔 `auto.commit.interval.ms`（默认 5s）自动提交当前 offset |
| Manual Commit | `enable.auto.commit=false` | 代码中显式调用 `commitSync()` 或 `commitAsync()` |

**Auto Commit 的风险**：
- Consumer poll 了一批消息，正在处理中，还没处理完，auto commit 触发了 → 如果此时 consumer 崩溃，这批消息不会被重新消费（**消息丢失/at-most-once**）
- 相反，如果处理完了但还没到 auto commit 时间就崩溃了 → 消息会被重新消费（**重复消费**）

**Manual Commit 的最佳实践**：
```
// 伪代码
while (true) {
    records = consumer.poll(Duration.ofMillis(100));
    for (record : records) {
        process(record);  // 先处理
    }
    consumer.commitSync();  // 处理完再提交 → at-least-once
}
```

**commitSync vs commitAsync**：
- `commitSync()`：阻塞等待 broker 确认，可靠但慢
- `commitAsync()`：非阻塞，快但如果失败不会自动重试（因为重试时 offset 可能已经过时）
- 常见组合：正常用 `commitAsync()`，consumer 关闭前用 `commitSync()` 确保最后的 offset 被提交

---



## 6. ZooKeeper → KRaft

### 🎤 English Answer (30-40s)

Kafka traditionally relied on ZooKeeper for cluster metadata management — things like broker registration, topic configs, partition leadership, and controller election. But ZooKeeper added operational complexity and was a scalability bottleneck, especially for large clusters with many partitions. KRaft — Kafka Raft — replaces ZooKeeper by embedding a Raft-based consensus protocol directly into Kafka. A set of controller nodes form a quorum that manages metadata using an internal metadata log. This simplifies deployment, improves startup time, and enables Kafka to scale to millions of partitions. ZooKeeper mode is deprecated as of Kafka 3.5 and planned for removal.

### 📖 中文详解

**为什么要替换 ZooKeeper**：
- **运维复杂**：需要单独部署和维护一套 ZooKeeper 集群，增加运维负担
- **性能瓶颈**：ZooKeeper 不擅长处理大量频繁的元数据更新，当 partition 数量达到几十万级别时，controller failover 可能需要几分钟
- **数据不一致**：Kafka 和 ZooKeeper 之间的元数据可能出现不一致
- **架构耦合**：升级一个系统需要考虑另一个系统的兼容性

**KRaft 模式的核心设计**：
- Controller quorum：若干节点组成 controller quorum（通常 3 个或 5 个），通过 Raft 协议选举出一个 active controller
- Metadata log：所有元数据变更（topic 创建、partition 分配、leader 变更等）被记录在一个内部的 metadata topic 中，像普通 Kafka log 一样是 append-only 的
- Event-driven：Broker 通过订阅 metadata log 获取最新状态，而不是像以前那样 controller 推送给所有 broker

**KRaft 的优势**：
- 启动速度快（从 metadata log 快速恢复状态）
- Controller failover 时间从分钟级降到秒级
- 理论上支持百万级 partition
- 部署更简单（只需要 Kafka 自身）

**迁移路径**：Kafka 3.3 开始 KRaft 进入 production-ready，Kafka 3.5 标记 ZooKeeper 为 deprecated，计划在 Kafka 4.0 彻底移除 ZooKeeper 支持。

---



## 7. Watermark (in Stream Processing)

### 🎤 English Answer (30-40s)

A watermark is a concept in stream processing that tracks the progress of **event time**. It's essentially a timestamp that says: "I believe all events with a timestamp earlier than this have already arrived." Watermarks are used to determine when to trigger window computations. For example, if you have a 10-minute tumbling window ending at 10:10, and the watermark passes 10:10, the system considers that window complete and emits results. Watermarks are typically generated from the data itself — often as the maximum observed event time minus some allowed lateness. They're essential for handling out-of-order data in systems like Flink, Spark Structured Streaming, and Kafka Streams.

### 📖 中文详解

**为什么需要 Watermark**：在流处理中，数据到达的顺序不一定等于事件发生的顺序（比如移动设备先离线再上线发送数据）。Processing time 和 event time 之间有差异。Watermark 就是用来告诉系统："event time 推进到哪里了"。

**Watermark 的定义**：
- Watermark(T) 表示系统认为不会再收到 event time < T 的数据
- 通常计算方式：`watermark = max(observed event time) - allowed lateness`
- 比如设置 allowed lateness = 5 分钟，当前看到的最大 event time 是 10:20，那么 watermark = 10:15

**Watermark 在 Window 计算中的作用**：
```
假设 10 分钟的 tumbling window: [10:00, 10:10)
- 数据持续到达，event time 在 10:00 ~ 10:12 之间
- 当 watermark 推进到 10:10 时，系统认为 [10:00, 10:10) 窗口内的数据都已到齐
- 触发窗口计算，输出结果
```

**不同框架的实现**：
- **Flink**：通过 `WatermarkStrategy` 配置，支持 periodic 和 punctuated 两种 watermark 生成方式
- **Spark Structured Streaming**：通过 `withWatermark("event_time", "10 minutes")` 设置
- **Kafka Streams**：基于 stream time（所有输入 partition 的最小时间戳）隐式推进

**Watermark 的权衡**：
- 设太小 → 很多 late data 被丢弃，结果不准确
- 设太大 → 窗口迟迟不关闭，延迟增加，state 膨胀

---



## 8. Late Data Handling / 迟到数据处理

### 🎤 English Answer (30-40s)

Late data refers to events that arrive after the watermark has already passed their event time, meaning their window has been closed. There are several strategies to handle this. First, you can set an **allowed lateness** — for example, in Flink you can keep a window open for extra time after the watermark passes, and update results when late data arrives. Second, some frameworks support **side outputs** where late data is redirected to a separate stream for special handling. Third, in Spark Structured Streaming, the watermark defines how long state is kept — data arriving within the watermark threshold triggers updates, but data arriving after is dropped. The right approach depends on your accuracy requirements versus latency and resource constraints.

### 📖 中文详解

**什么是 Late Data**：当 watermark 已经推进到时间 T，此时到达一条 event time < T 的数据，这条数据就是"late data"。它对应的窗口可能已经计算并输出结果了。

**处理策略**：

**策略一：设置 Allowed Lateness（允许延迟）**
- Flink 中：`window.allowedLateness(Time.minutes(5))`
- 窗口在 watermark 触发后不立即销毁，保留一段时间
- 迟到数据到达后，重新触发窗口计算，输出更新结果
- 缺点：需要维护更多 state，占用更多内存

**策略二：Side Output（侧输出）**
- Flink 中：通过 `OutputTag` 把迟到数据路由到单独的流
- 后续可以对迟到数据做特殊处理（写入数据库修正、告警等）
```
// Flink 伪代码
OutputTag<Event> lateTag = new OutputTag<>("late-data");
result.getSideOutput(lateTag);  // 获取迟到数据流
```

**策略三：直接丢弃**
- 超过 allowed lateness 的数据直接丢弃
- 适用于对精确度要求不高的场景（如实时监控大屏）

**策略四：Accumulation Mode（累积模式）**
- Flink / Beam 支持不同的 accumulation mode：
  - **Discarding**：每次触发只输出增量
  - **Accumulating**：每次触发输出全量累积结果
  - **Accumulating & Retracting**：输出新结果 + 撤回旧结果

**Spark Structured Streaming 的处理**：
- 通过 `withWatermark()` 控制 state 保留时长
- 在 watermark 范围内的迟到数据会参与更新（append/update mode）
- 超过 watermark 的数据被 drop，对应 state 被清理

---



## 9. State Store / 状态存储

### 🎤 English Answer (30-40s)

In stream processing, state stores hold intermediate computation results — like running aggregations, window contents, or join buffers. In **Kafka Streams**, the default state store is **RocksDB**, an embedded key-value store that persists state locally on disk. It also maintains a changelog topic in Kafka for fault tolerance — if a node fails, the state can be rebuilt from the changelog. In **Flink**, state is managed by a state backend. The **HashMapStateBackend** keeps state in JVM heap memory, suitable for small state. The **EmbeddedRocksDBStateBackend** stores state on local disk, enabling much larger state. Both support checkpointing to durable storage like S3 or HDFS for recovery.

### 📖 中文详解

**为什么需要 State Store**：流处理中很多操作是有状态的（stateful），比如聚合（count/sum）、窗口计算、Join 等都需要记住之前的数据。State store 就是存放这些中间状态的地方。

**Kafka Streams 的 State Store**：
- 默认使用 **RocksDB**（嵌入式 key-value 数据库，基于 LSM tree）
- State 存储在本地磁盘上，访问速度快
- 每个 state store 对应一个 Kafka **changelog topic**（compact log），每次 state 变更都会写入
- 如果节点宕机，新节点可以从 changelog topic 完全恢复 state
- 也支持 in-memory state store（纯内存，快但 state 量受限）
- State store 可以被 Interactive Queries 暴露为可查询的 API

**Flink 的 State Backend**：

| State Backend | 存储位置 | 适用场景 |
|--------------|---------|---------|
| HashMapStateBackend | JVM heap 内存 | State 小、延迟要求极低 |
| EmbeddedRocksDBStateBackend | 本地磁盘（RocksDB） | State 大、生产环境首选 |

- Flink 的 state 分为 **Keyed State**（按 key 分区的状态，如 ValueState、ListState、MapState）和 **Operator State**（与 operator 实例绑定的状态，如 Kafka consumer offset）
- 通过 **checkpoint** 机制定期将 state 快照到远程存储（S3/HDFS）

**Spark Structured Streaming 的 State**：
- State 存储在 executor 内存中
- 通过 checkpoint 目录（通常是 HDFS/S3）持久化
- `groupBy().agg()` 和 `flatMapGroupsWithState()` 等操作会维护 state

---



## 10. Checkpoint / 检查点

### 🎤 English Answer (30-40s)

Checkpointing is the mechanism for achieving fault tolerance in stream processing. It periodically captures a consistent snapshot of the entire application state — including operator states, offsets, and in-flight data — and saves it to durable storage like HDFS or S3. If a failure occurs, the application can restart from the latest checkpoint rather than reprocessing everything from the beginning. In Flink, checkpointing uses the **Chandy-Lamport algorithm** — the system injects **barrier markers** into the data stream, and when an operator receives barriers from all its inputs, it snapshots its state. This allows checkpointing without stopping data processing. Combined with Kafka's offset tracking, checkpointing enables exactly-once semantics end to end.

### 📖 中文详解

**Checkpoint 的核心目的**：保证流处理应用在发生故障后能够从一个一致的状态恢复，而不是从头开始重新处理所有数据。

**Flink 的 Checkpoint 机制（重点！）**：

**Chandy-Lamport 分布式快照算法**：
1. **JobManager** 定期向所有 source operator 注入 **checkpoint barrier**（一种特殊标记）
2. Barrier 随着数据流向下游传播
3. 当一个 operator 收到所有输入通道的 barrier 后：
   - 对当前 state 做快照
   - 将快照异步写入远程存储（S3/HDFS）
   - 把 barrier 继续向下游发送
4. 当所有 operator 都完成快照，这次 checkpoint 就成功了

**Barrier Alignment（对齐）**：
- 如果一个 operator 有多个输入流，它需要等待所有输入的 barrier 都到齐
- 在等待期间，先到达 barrier 的输入通道的后续数据会被 buffer
- Flink 1.11+ 支持 **Unaligned Checkpoint**：不用等待对齐，直接快照，把还在 buffer 中的数据也一起存入 checkpoint，减少背压时的 checkpoint 延迟

**Checkpoint 配置关键参数**：
- `checkpoint interval`：多久做一次 checkpoint（如 1 分钟）
- `checkpoint timeout`：单次 checkpoint 超时时间
- `min pause between checkpoints`：两次 checkpoint 之间的最小间隔
- `max concurrent checkpoints`：允许同时进行的 checkpoint 数
- `externalized checkpoints`：是否保留 checkpoint 供手动恢复（建议开启）

**Checkpoint vs Savepoint**：
- **Checkpoint**：系统自动触发，用于故障恢复，可能会被系统自动清理
- **Savepoint**：手动触发，用于版本升级、代码变更、集群迁移等有计划的操作

**Spark Structured Streaming 的 Checkpoint**：
- 每个 micro-batch 结束后将 offset 和 state 写入 checkpoint 目录
- 结构包括：`offsets/`（每个 batch 读取的 offset）、`commits/`（完成的 batch）、`state/`（状态快照）
- 恢复时从 checkpoint 目录读取上次的进度继续处理

**Kafka Streams 的 Checkpoint**：
- 不使用传统 checkpoint，而是通过 **changelog topic** 实现 state 恢复
- 本地 state store 的 `.checkpoint` 文件记录恢复到哪个 offset，避免不必要的重放

---



## 11. 补充高频知识点

### Consumer Group Rebalance

**English**: When consumers join or leave a group, Kafka triggers a rebalance to redistribute partitions. During rebalance, the group is temporarily unavailable. Strategies include Eager (stop-the-world), Cooperative Sticky (incremental, minimizes partition movement), and Static Group Membership (reduces unnecessary rebalances).

**中文**：当 consumer group 的成员数量变化时（加入、离开、崩溃），Kafka 触发 rebalance 重新分配 partition。Rebalance 期间所有 consumer 停止消费。Cooperative Sticky Assignor 可以实现增量 rebalance，减少影响范围。

### Kafka Partitioning Strategy

**English**: By default, messages with a key are hashed to a partition (murmur2), messages without a key use sticky partitioning (batches to one partition, then switches). Custom partitioners can implement business-specific routing.

**中文**：有 key 的消息通过 hash（murmur2）决定 partition，保证相同 key 的消息进入同一个 partition（保序）。没有 key 的消息使用 sticky partitioning（粘性分区，对同一 batch 的消息发往同一 partition，减少请求数）。

### Backpressure

**English**: Backpressure occurs when downstream operators can't keep up with the data rate. Flink handles it naturally through its network buffer pool — slow operators cause buffers to fill up, which slows down upstream. Kafka acts as a natural backpressure buffer between producers and consumers.

**中文**：当下游处理速度跟不上上游时产生 backpressure。Flink 通过 network buffer 机制自然传导背压——下游慢了，buffer 满了，上游自动变慢。Kafka 本身作为 broker 也是一个天然的缓冲区，解耦了 producer 和 consumer 的速度差异。

---



## 12. 速查对比表

| 概念 | Kafka | Flink | Spark Streaming |
|------|-------|-------|----------------|
| State 存储 | RocksDB + Changelog Topic | RocksDB / Heap + Checkpoint to S3 | Executor Memory + Checkpoint Dir |
| Exactly-Once | Idempotent Producer + Transactions | Checkpoint + Two-Phase Commit Sink | Structured Streaming + Idempotent Sink |
| 时间语义 | Event time / Processing time | Event time / Processing time / Ingestion time | Event time / Processing time |
| 迟到数据 | N/A（Kafka 本身不处理窗口） | Allowed Lateness + Side Output | Watermark 机制自动 drop |
| Fault Tolerance | Replication + ISR | Checkpoint (Chandy-Lamport) | Checkpoint + WAL (DStream) / Checkpoint (Structured) |
| 延迟 | 毫秒级（纯消息传递） | 毫秒~秒级（真正的流） | 秒~分钟级（micro-batch） |

